In [ ]:
# Standard libraries
from pathlib import Path
import os
import json
import re

# Data processing
import pandas as pd
import numpy as np
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pypdf import PdfReader


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVALUATION_DIR = DATA_DIR / "evaluation"

CSV_PATH = RAW_DIR / "Resume" / "Resume.csv"
PDF_DIR = RAW_DIR / "data" / "data"

# Create output directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("CSV exists:", CSV_PATH.exists())
print("PDF directory exists:", PDF_DIR.exists())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

display(df.head())

In [ ]:
missing_values = df.isnull().sum()

missing_report = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (
        missing_values / len(df)
    ) * 100
})

display(
    missing_report.sort_values(
        by="Missing_Percentage",
        ascending=False
    )
)


print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate resume texts:",
    df["Resume_str"].duplicated().sum()
)

In [ ]:
category_counts = df["Category"].value_counts()

print("Total categories:", df["Category"].nunique())

display(category_counts)

plt.figure(figsize=(12, 6))

sns.countplot(
    data=df,
    y="Category",
    order=category_counts.index
)

plt.title("Resume Distribution by Category")
plt.xlabel("Number of Resumes")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
df["char_count"] = df["Resume_str"].fillna("").str.len()

df["word_count"] = (
    df["Resume_str"]
    .fillna("")
    .str.split()
    .str.len()
)

display(
    df[["char_count", "word_count"]].describe()
)


plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="word_count",
    bins=50,
    kde=True
)

plt.title("Resume Word Count Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.show()

In [ ]:
duplicate_mask = df["Resume_str"].duplicated(keep=False)

duplicate_resumes = df.loc[
    duplicate_mask,
    ["ID", "Category", "Resume_str"]
]

print("Duplicate resume records:", len(duplicate_resumes))

display(
    duplicate_resumes[["ID", "Category"]]
)

In [ ]:
duplicate_groups = (
    df[df["Resume_str"].duplicated(keep=False)]
    .groupby("Resume_str")
)

for resume_text, group in duplicate_groups:
    print("Number of occurrences:", len(group))
    print("IDs:", group["ID"].tolist())
    print("Categories:", group["Category"].tolist())
    print("-" * 60)

In [ ]:
empty_resumes = (
    df["Resume_str"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

short_resumes = (df["word_count"] < 50).sum()

long_resumes = (df["word_count"] > 3000).sum()

print("Empty resumes:", empty_resumes)
print("Resumes below 50 words:", short_resumes)
print("Resumes above 3000 words:", long_resumes)

In [ ]:
# Inspect empty and short resumes
problematic_resumes = df[
    df["word_count"] < 50
]

for _, row in problematic_resumes.iterrows():
    print("ID:", row["ID"])
    print("Category:", row["Category"])
    print("Word count:", row["word_count"])
    print("Resume text:", repr(row["Resume_str"][:500]))
    print("-" * 60)

In [ ]:
long_resumes = df.nlargest(
    7,
    "word_count"
)

display(
    long_resumes[
        ["ID", "Category", "char_count", "word_count"]
 
    ]
)

In [ ]:
for index in range(3):
    print(f"\nRecord {index + 1}")

    print("Plain text:")
    print(df.loc[index, "Resume_str"][:300])

    print("\nHTML:")
    print(str(df.loc[index, "Resume_html"])[:500])

    print("=" * 80)

In [ ]:
# Work on a copy. Never modify the original raw DataFrame.
clean_df = df.copy()

# Remove empty or whitespace-only resumes
clean_df = clean_df[
    clean_df["Resume_str"].fillna("").str.strip().ne("")
].copy()

# Remove exact duplicate resume texts
clean_df = clean_df.drop_duplicates(
    subset=["Resume_str"],
    keep="first"
).copy()

print("Original records:", len(df))
print("Cleaned records:", len(clean_df))
print("Removed records:", len(df) - len(clean_df))

print("Remaining empty resumes:",
      clean_df["Resume_str"].fillna("").str.strip().eq("").sum())

print("Remaining duplicate texts:",
      clean_df["Resume_str"].duplicated().sum())

In [ ]:
CLEANED_PATH = PROCESSED_DIR / "resumes_cleaned.csv"

clean_df.to_csv(CLEANED_PATH, index=False)

print("Saved:", CLEANED_PATH)

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

def extract_html_text(html):
    if pd.isna(html):
        return ""

    soup = BeautifulSoup(html, "html.parser")

    return soup.get_text(
        separator="\n",
        strip=True
    )

In [ ]:
sample = clean_df.head(5).copy()

sample["html_extracted"] = (
    sample["Resume_html"].apply(extract_html_text)
)

for _, row in sample.iterrows():

    print("Resume ID:", row["ID"])

    print("\nOriginal plain text:")
    print(row["Resume_str"][:500])

    print("\nText extracted from HTML:")
    print(row["html_extracted"][:500])

    print("\nPlain text length:", len(row["Resume_str"]))
    print("HTML text length:", len(row["html_extracted"]))

    print("=" * 80)

In [ ]:
clean_df["html_extracted"] = (
    clean_df["Resume_html"].apply(extract_html_text)
)

clean_df["html_word_count"] = (
    clean_df["html_extracted"].str.split().str.len()
)

clean_df["plain_word_count"] = (
    clean_df["Resume_str"].str.split().str.len()
)

print(
    "Empty HTML extractions:",
    clean_df["html_extracted"].str.strip().eq("").sum()
)

print(
    clean_df[
        ["plain_word_count", "html_word_count"]
    ].describe()
)

In [ ]:
from pathlib import Path
from collections import Counter

PDF_DIR = RAW_DIR / "data" / "data"

pdf_files = list(PDF_DIR.rglob("*.pdf"))

print("Total PDF files:", len(pdf_files))

category_counts = Counter(
    pdf.parent.name for pdf in pdf_files
)

print("\nPDF count by category:")

for category, count in category_counts.most_common():
    print(category, ":", count)

In [ ]:
import re
from difflib import SequenceMatcher

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def calculate_similarity(row):
    plain = normalize_text(row["Resume_str"])
    html = normalize_text(row["html_extracted"])

    return SequenceMatcher(
        None,
        plain,
        html
    ).ratio()


clean_df["text_similarity"] = clean_df.apply(
    calculate_similarity,
    axis=1
)

print(clean_df["text_similarity"].describe())

In [ ]:
display(
    clean_df.nsmallest(
        10,
        "text_similarity"
    )[
        ["ID", "Category", "text_similarity"]
    ]
)

In [ ]:
sample_pdf = pdf_files[1]

reader = PdfReader(str(sample_pdf))

extracted_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

print("PDF:", sample_pdf.name)
print("Category:", sample_pdf.parent.name)
print("Pages:", len(reader.pages))
print("Extracted characters:", len(extracted_text))

print("\nExtracted text:")
print(extracted_text[:1500])

In [ ]:
from pathlib import Path
from pypdf import PdfReader
import pandas as pd
import time

PDF_DIR = RAW_DIR / "data" / "data"

OUTPUT_PATH = PROCESSED_DIR / "pdf_extraction_results.csv"

# Find all PDF files, including uppercase .PDF extensions
pdf_files = [
    path
    for path in PDF_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
]

print("Total PDF files:", len(pdf_files))

In [ ]:
def extract_pdf_text(pdf_path):

    try:
        reader = PdfReader(str(pdf_path), strict=False)

        # Check encryption
        if reader.is_encrypted:
            return {
                "page_count": len(reader.pages),
                "extracted_text": "",
                "char_count": 0,
                "status": "encrypted"
            }

        pages_text = []

        for page in reader.pages:

            text = page.extract_text() or ""

            pages_text.append(text)

        extracted_text = "\n".join(pages_text)

        char_count = len(extracted_text.strip())

        if char_count == 0:
            status = "empty_extraction"

        elif char_count < 100:
            status = "low_text"

        else:
            status = "success"

        return {
            "page_count": len(reader.pages),
            "extracted_text": extracted_text,
            "char_count": char_count,
            "status": status
        }

    except Exception as e:

        return {
            "page_count": 0,
            "extracted_text": "",
            "char_count": 0,
            "status": f"error: {type(e).__name__}: {str(e)[:150]}"
        }

In [ ]:
results = []

start_time = time.time()

for index, pdf_path in enumerate(pdf_files, start=1):

    result = extract_pdf_text(pdf_path)

    results.append({
        "filename": pdf_path.name,
        "file_path": str(pdf_path),
        "category": pdf_path.parent.name,
        "page_count": result["page_count"],
        "char_count": result["char_count"],
        "status": result["status"],
        "extracted_text": result["extracted_text"]
    })

    if index % 100 == 0:
        print(f"Processed {index}/{len(pdf_files)} PDFs")

end_time = time.time()

print("\nBatch extraction completed!")
print("Total PDFs:", len(pdf_files))
print("Processing time:", round(end_time - start_time, 2), "seconds")

In [ ]:
pdf_results_df = pd.DataFrame(results)

print("Results shape:", pdf_results_df.shape)

display(pdf_results_df.head())
print(pdf_results_df["status"].value_counts())


total = len(pdf_results_df)

successful = (
    pdf_results_df["status"] == "success"
).sum()

failed = total - successful

print("Total PDFs:", total)
print("Successful:", successful)
print("Failed:", failed)

print(
    "Success rate:",
    round((successful / total) * 100, 2),
    "%"
)

In [ ]:
failed_pdfs = pdf_results_df[
    pdf_results_df["status"] != "success"
]

display(
    failed_pdfs[
        ["filename", "category", "page_count",
         "char_count", "status"]
    ]
)

from pypdf import PdfReader

failed_path = failed_pdfs.iloc[0]["file_path"]

reader = PdfReader(failed_path, strict=False)

print("Pages:", len(reader.pages))

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    print(f"Page {i+1}: {len(text)} characters")

In [ ]:
OUTPUT_PATH = PROCESSED_DIR / "pdf_extraction_results.csv"

pdf_results_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

In [ ]:
from pypdf import PdfReader

failed_path = failed_pdfs.iloc[0]["file_path"]

reader = PdfReader(failed_path, strict=False)

print("Metadata:", reader.metadata)
print("Page count:", len(reader.pages))

page = reader.pages[0]

print("Page dimensions:", page.mediabox)
print("Extracted text:", repr(page.extract_text()))
print("Images:", len(page.images))

In [ ]:
successful_pdfs = pdf_results_df[
    pdf_results_df["status"] == "success"
]

sample = successful_pdfs.iloc[0]

print("Filename:", sample["filename"])
print("Category:", sample["category"])
print("Pages:", sample["page_count"])

print("\nExtracted resume:\n")
print(sample["extracted_text"])

In [ ]:
sample_categories = (
    successful_pdfs["category"]
    .drop_duplicates()
    .head(5)
    .tolist()
)

for category in sample_categories:

    sample = successful_pdfs[
        successful_pdfs["category"] == category
    ].iloc[0]

    print("\n" + "=" * 70)
    print("CATEGORY:", category)
    print("FILENAME:", sample["filename"])
    print("PAGES:", sample["page_count"])
    print("CHARACTERS:", sample["char_count"])
    print("=" * 70)

    print(sample["extracted_text"][:1500])

In [ ]:
# Get five different categories
sample_categories = sorted(
    successful_pdfs["category"].dropna().unique()
)[:5]

print("Selected categories:", sample_categories)
print("Number of categories:", len(sample_categories))

# Inspect one PDF from each category
for category in sample_categories:

    sample = successful_pdfs[
        successful_pdfs["category"] == category
    ].iloc[0]

    print("\n" + "=" * 70)
    print("CATEGORY:", category)
    print("FILENAME:", sample["filename"])
    print("PAGES:", sample["page_count"])
    print("CHARACTERS:", sample["char_count"])
    print("=" * 70)

    print(sample["extracted_text"][:1000])

In [ ]:
import re
import html

all_text = (
    successful_pdfs["extracted_text"]
    .fillna("")
    .astype(str)
)

# Count HTML entities
html_entity_count = all_text.str.count(
    r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z]+);"
).sum()

# Count suspicious encoding characters
suspicious_chars = ["Â", "Ã", "ï", "�"]

encoding_count = sum(
    all_text.str.count(re.escape(char)).sum()
    for char in suspicious_chars
)

# Count repeated spaces and tabs
multiple_spaces = all_text.str.count(
    r"[ \t]{2,}"
).sum()

print("HTML entities:", html_entity_count)
print("Potential encoding artifacts:", encoding_count)
print("Repeated spaces/tabs:", multiple_spaces)

In [ ]:
suspicious_chars = ["Â", "Ã", "ï", "�"]

for char in suspicious_chars:

    count = all_text.str.count(
        re.escape(char)
    ).sum()

    print(repr(char), ":", count)


for char in suspicious_chars:

    mask = all_text.str.contains(
        re.escape(char),
        na=False
    )

    print("\nCharacter:", repr(char))
    print("Affected resumes:", mask.sum())

    examples = all_text[mask].head(3)

    for text in examples:
        position = text.find(char)
        print(text[max(0, position-40):position+60])

In [ ]:
test_texts = [
    "CafÃ©",
    "EntrÃ©es",
    "Accounting for six companiesÂ",
    "Designed Â weekly lesson plans",
    "ï¼​"
]

for text in test_texts:

    print("\nOriginal:", repr(text))

    try:
        # Remove zero-width spaces only for this diagnostic test
        test = text.replace("\u200b", "")

        repaired = test.encode(
            "latin-1"
        ).decode("utf-8")

        print("Repaired:", repr(repaired))

    except UnicodeError as e:
        print("Cannot repair:", e)

In [ ]:
import re
import html
import unicodedata

def clean_resume_text(text):

    if not isinstance(text, str):
        return ""

    # 1. Decode HTML entities
    text = html.unescape(text)

    # 2. Remove zero-width formatting characters
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")

    # 3. Normalize Unicode representation
    text = unicodedata.normalize("NFC", text)

    # 4. Normalize spaces and tabs without removing newlines
    text = re.sub(r"[ \t]+", " ", text)

    # 5. Remove spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    # 6. Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
test_text = """
Accounting for six companiesÂ
Skills     Python    Java
Designed weekly lesson plans&#x20;
"""

cleaned_test = clean_resume_text(test_text)

print("Original:")
print(repr(test_text))

print("\nCleaned:")
print(repr(cleaned_test))

In [ ]:
pdf_results_df["cleaned_text"] = (
    pdf_results_df["extracted_text"]
    .apply(clean_resume_text)
)

In [ ]:
empty_after_cleaning = (
    pdf_results_df["cleaned_text"]
    .str.strip()
    .eq("")
)

print("Empty after cleaning:", empty_after_cleaning.sum())

In [ ]:
pdf_results_df["original_char_count"] = (
    pdf_results_df["extracted_text"].str.len()
)

pdf_results_df["cleaned_char_count"] = (
    pdf_results_df["cleaned_text"].str.len()
)

display(
    pdf_results_df[
        ["filename", "original_char_count", "cleaned_char_count"]
    ].head(10)
)

In [ ]:
sample = pdf_results_df[
    pdf_results_df["status"] == "success"
].iloc[0]

print(sample["cleaned_text"][:2000])

In [ ]:
sample_text = pdf_results_df.loc[
    pdf_results_df["filename"] == "10554236.pdf",
    "cleaned_text"
].iloc[0]

print("Remaining HTML entities:")

matches = re.findall(
    r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z]+);",
    sample_text
)

print(matches[:20])
print("Total remaining:", len(matches))

In [ ]:
original_text = pdf_results_df.loc[
    pdf_results_df["filename"] == "10554236.pdf",
    "extracted_text"
].iloc[0]

print("Original:", original_text.count("&#x20;"))
print("Cleaned:", sample_text.count("&#x20;"))

In [ ]:
entity_pattern = (
    r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z]+);"
)

remaining_entities = (
    pdf_results_df["cleaned_text"]
    .fillna("")
    .str.count(entity_pattern)
)

print("Remaining HTML entities:", remaining_entities.sum())

print(
    "PDFs containing HTML entities:",
    (remaining_entities > 0).sum()
)

In [ ]:
for char in ["Â", "Ã", "ï", "�"]:

    count = (
        pdf_results_df["cleaned_text"]
        .fillna("")
        .str.count(re.escape(char))
        .sum()
    )

    print(repr(char), ":", count)

In [ ]:
entity_pattern = (
    r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z]+);"
)

for _, row in pdf_results_df.iterrows():

    text = row["cleaned_text"] or ""

    matches = re.findall(entity_pattern, text)

    if matches:
        print("\nFilename:", row["filename"])
        print("Category:", row["category"])
        print("Entities:", matches)

In [ ]:
for _, row in pdf_results_df.iterrows():

    text = row["cleaned_text"] or ""

    matches = list(
        re.finditer(entity_pattern, text)
    )

    for match in matches:

        start = max(0, match.start() - 80)
        end = min(len(text), match.end() + 80)

        print("\nFilename:", row["filename"])
        print("Category:", row["category"])
        print("Matched text:", match.group())
        print("Context:", repr(text[start:end]))

In [ ]:
import re
import html

entity_pattern = r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z][A-Za-z0-9]+);"

def count_decodable_entities(text):
    if not isinstance(text, str):
        return 0

    matches = re.findall(entity_pattern, text)

    return sum(
        html.unescape(match) != match
        for match in matches
    )

remaining = pdf_results_df["cleaned_text"].apply(
    count_decodable_entities
).sum()

print("Remaining recognized HTML entities:", remaining)



In [ ]:
OUTPUT_PATH = PROCESSED_DIR / "pdf_extraction_results.csv"

pdf_results_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:", OUTPUT_PATH)
print("Records:", len(pdf_results_df))

In [ ]:
csv_ids = set(
    df["ID"].astype(str).str.replace(r"\.0$", "", regex=True)
)

pdf_ids = set(
    pdf_results_df["filename"].str.replace(
        r"\.pdf$", "", regex=True
    )
)

print("CSV unique IDs:", len(csv_ids))
print("PDF unique IDs:", len(pdf_ids))
print("Matching IDs:", len(csv_ids & pdf_ids))
print("CSV IDs without PDFs:", len(csv_ids - pdf_ids))
print("PDF IDs without CSV records:", len(pdf_ids - csv_ids))

In [ ]:
# Create PDF ID from filename
pdf_results_df["ID"] = (
    pdf_results_df["filename"]
    .str.replace(".pdf", "", regex=False)
)

# Convert IDs to a consistent type
df["ID"] = df["ID"].astype(str).str.replace(
    r"\.0$", "", regex=True
)

pdf_results_df["ID"] = pdf_results_df["ID"].astype(str)

# Merge CSV and PDF datasets
comparison_df = df.merge(
    pdf_results_df[["ID", "cleaned_text", "status"]],
    on="ID",
    how="inner",
    validate="one_to_one"
)

print("Comparison dataset shape:", comparison_df.shape)

display(
    comparison_df[
        ["ID", "Category", "Resume_str", "cleaned_text"]
    ].head()
)

In [ ]:
import re

def normalize_text(text):

    text = str(text).lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove punctuation, preserving letters and numbers
    text = re.sub(r"[^\w\s]", "", text)

    return text.strip()

In [ ]:
from difflib import SequenceMatcher

def calculate_similarity(row):

    csv_text = normalize_text(row["Resume_str"])
    pdf_text = normalize_text(row["cleaned_text"])

    if not csv_text or not pdf_text:
        return 0.0

    return SequenceMatcher(
        None,
        csv_text,
        pdf_text
    ).ratio()


comparison_df["similarity"] = comparison_df.apply(
    calculate_similarity,
    axis=1
)

print(
    comparison_df["similarity"].describe()
)


exact_matches = (
    comparison_df["similarity"] == 1.0
).sum()

print("Total records:", len(comparison_df))
print("Exact matches:", exact_matches)
print("Non-exact matches:", len(comparison_df) - exact_matches)

In [ ]:
def categorize_similarity(score):

    if score == 1.0:
        return "Exact Match"

    elif score >= 0.90:
        return "High Similarity"

    elif score >= 0.70:
        return "Moderate Similarity"

    else:
        return "Low Similarity"


comparison_df["similarity_category"] = (
    comparison_df["similarity"]
    .apply(categorize_similarity)
)

print(
    comparison_df["similarity_category"].value_counts()
)



mismatched = comparison_df[
    comparison_df["similarity"] < 1.0
].sort_values("similarity")

display(
    mismatched[
        ["ID", "Category", "similarity",
         "Resume_str", "cleaned_text"]
    ].head(5)
)

In [ ]:
# Exclude records where both sources have empty text
comparison_df["csv_normalized"] = (
    comparison_df["Resume_str"].apply(normalize_text)
)

comparison_df["pdf_normalized"] = (
    comparison_df["cleaned_text"].apply(normalize_text)
)

# Identify empty records
comparison_df["is_empty"] = (
    comparison_df["csv_normalized"].eq("") &
    comparison_df["pdf_normalized"].eq("")
)

# Get records with similarity below 0.90
mismatch_df = comparison_df[
    (comparison_df["similarity"] < 0.90) &
    (~comparison_df["is_empty"])
].copy()

print("Records requiring investigation:", len(mismatch_df))

display(
    mismatch_df[
        [
            "ID",
            "Category",
            "similarity",
            "status"
        ]
    ].sort_values("similarity")
)

In [ ]:
mismatch_df["csv_length"] = (
    mismatch_df["csv_normalized"].str.len()
)

mismatch_df["pdf_length"] = (
    mismatch_df["pdf_normalized"].str.len()
)

mismatch_df["length_difference"] = (
    mismatch_df["csv_length"] -
    mismatch_df["pdf_length"]
)

display(
    mismatch_df[
        [
            "ID",
            "similarity",
            "csv_length",
            "pdf_length",
            "length_difference"
        ]
    ].sort_values("similarity")
)

In [ ]:
from difflib import unified_diff

inspect_ids = [
    "14663897",
    "12386670",
    "15746146",
    "23477199"
]

for resume_id in inspect_ids:

    row = comparison_df[
        comparison_df["ID"] == resume_id
    ].iloc[0]

    csv_text = row["csv_normalized"]
    pdf_text = row["pdf_normalized"]

    print("\n" + "=" * 80)
    print("ID:", resume_id)
    print("Category:", row["Category"])
    print("Similarity:", row["similarity"])
    print("CSV length:", len(csv_text))
    print("PDF length:", len(pdf_text))

    print("\nCSV TEXT (first 1000 characters):")
    print(csv_text[:1000])

    print("\nPDF TEXT (first 1000 characters):")
    print(pdf_text[:1000])

    print("\n" + "-" * 80)

In [ ]:
from pypdf import PdfReader

pdf_path = r"C:\Users\vishn\OneDrive\Desktop\LLM\data\raw\data\data\CHEF\14663897.pdf"

reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

total_chars = 0

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text() or ""
    total_chars += len(page_text)

    print(f"\n{'=' * 60}")
    print(f"PAGE {page_number}")
    print("Characters:", len(page_text))

    print("\nSTART:")
    print(repr(page_text[:200]))

    print("\nEND:")
    print(repr(page_text[-200:]))

print("\nTotal extracted characters:", total_chars)

In [ ]:
resume_id = "12386670"

row = comparison_df[
    comparison_df["ID"] == resume_id
].iloc[0]

print("ID:", row["ID"])
print("Category:", row["Category"])

print("\nCSV:")
print(row["Resume_str"][:1500])

print("\nPDF:")
print(row["cleaned_text"][:1500])

In [ ]:
from difflib import SequenceMatcher

resume_id = "14663897"

row = comparison_df[
    comparison_df["ID"] == resume_id
].iloc[0]

csv_text = row["csv_normalized"]
pdf_text = row["pdf_normalized"]

chunk_size = 500

results = []

for start in range(0, len(csv_text), chunk_size):

    chunk = csv_text[start:start + chunk_size]

    similarity = SequenceMatcher(
        None,
        chunk,
        pdf_text
    ).ratio()

    results.append({
        "start_position": start,
        "chunk_length": len(chunk),
        "similarity": similarity,
        "matched": chunk in pdf_text
    })

chunk_df = pd.DataFrame(results)

display(chunk_df)

In [ ]:
print(
    "Chunks found:",
    chunk_df["matched"].sum()
)

print(
    "Total chunks:",
    len(chunk_df)
)

print(
    "Chunk coverage:",
    round(chunk_df["matched"].mean() * 100, 2),
    "%"
)

In [ ]:
%pip install rapidfuzz

In [ ]:
from rapidfuzz import fuzz
import pandas as pd

chunk_results = []

for start in range(0, len(csv_text), 500):

    chunk = csv_text[start:start + 500]

    score = fuzz.partial_ratio(
        chunk,
        pdf_text
    )

    chunk_results.append({
        "start_position": start,
        "chunk_length": len(chunk),
        "partial_similarity": score,
        "matched": score >= 95
    })

chunk_df = pd.DataFrame(chunk_results)

display(chunk_df)

print(
    "Chunks with at least 95% partial similarity:",
    chunk_df["matched"].sum()
)

print(
    "Total chunks:",
    len(chunk_df)
)

print(
    "Approximate chunk coverage:",
    round(chunk_df["matched"].mean() * 100, 2),
    "%"
)

In [ ]:
unmatched_chunks = chunk_df[
    chunk_df["partial_similarity"] < 95
].copy()

display(unmatched_chunks)

In [ ]:
for _, item in unmatched_chunks.iterrows():

    start = int(item["start_position"])
    length = int(item["chunk_length"])

    chunk = csv_text[start:start + length]

    print("\n" + "=" * 80)
    print("Position:", start)
    print("Similarity:", item["partial_similarity"])
    print("\nCSV CHUNK:")
    print(chunk)

In [ ]:
from rapidfuzz import fuzz

pdf_in_csv = fuzz.partial_ratio(
    pdf_text,
    csv_text
)

print("PDF-to-CSV partial similarity:", pdf_in_csv)

In [ ]:
from rapidfuzz import fuzz

score = fuzz.ratio(
    pdf_text,
    csv_text
)

print("Full text similarity:", score)

In [ ]:
from difflib import SequenceMatcher

matcher = SequenceMatcher(
    None,
    csv_text,
    pdf_text,
    autojunk=False
)

for block in matcher.get_matching_blocks():
    if block.size > 0:
        print(
            f"CSV position: {block.a}, "
            f"PDF position: {block.b}, "
            f"Length: {block.size}"
        )

In [ ]:
matching_blocks = matcher.get_matching_blocks()

matched_chars = sum(
    block.size for block in matching_blocks
)

print("CSV length:", len(csv_text))
print("PDF length:", len(pdf_text))
print("Matched characters:", matched_chars)

print(
    "PDF matching coverage:",
    round(matched_chars / len(pdf_text) * 100, 2),
    "%"
)

print(
    "CSV matching coverage:",
    round(matched_chars / len(csv_text) * 100, 2),
    "%"
)

In [ ]:
import pandas as pd
from pathlib import Path

# Load cleaned dataset
DATA_PATH = Path("../data/processed/resumes_cleaned.csv")
df = pd.read_csv(DATA_PATH)

required_columns = ["ID", "Resume_str", "Category"]

missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Basic validation
if len(df) < 50:
    raise ValueError("Dataset contains fewer than 50 resumes.")

if df[required_columns].isnull().any().any():
    raise ValueError("Required columns contain missing values.")

# Calculate category counts
category_counts = df["Category"].value_counts()

# Calculate proportional allocation
raw_allocation = category_counts / len(df) * 50

allocation = raw_allocation.apply(int)

# Distribute remaining slots using largest fractional remainders
remaining = 50 - allocation.sum()

remainders = (raw_allocation - allocation).sort_values(
    ascending=False
)

for category in remainders.index:
    if remaining == 0:
        break

    if allocation[category] < category_counts[category]:
        allocation[category] += 1
        remaining -= 1

# Stratified sampling
sample_parts = []

for category, count in allocation.items():
    if count > 0:
        category_data = df[df["Category"] == category]

        selected = category_data.sample(
            n=int(count),
            random_state=42
        )

        sample_parts.append(selected)

sample = (
    pd.concat(sample_parts)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

# Verify sample size
assert len(sample) == 50
assert sample["ID"].is_unique

# Define candidate schema
fields = [
    "current_title",
    "previous_titles",
    "current_company",
    "previous_companies",
    "years_experience",
    "seniority",
    "primary_domain",
    "industries",
    "core_skills",
    "secondary_skills",
    "tools",
    "leadership_experience",
    "key_achievements",
    "location",
    "summary"
]

# Create annotation template
annotation_df = sample[
    ["ID", "Category", "Resume_str"]
].copy()

for field in fields:
    annotation_df[field] = None

# Save
OUTPUT_PATH = Path("../data/evaluation/resume_annotation_template.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

annotation_df.to_csv(OUTPUT_PATH, index=False)

print("Evaluation sample created successfully!")
print("Total resumes:", len(annotation_df))
print("Categories represented:", annotation_df["Category"].nunique())
print("Saved to:", OUTPUT_PATH)

In [ ]:
print("Shape:", annotation_df.shape)

print("\nCategory distribution:")
print(annotation_df["Category"].value_counts())

display(annotation_df.head(3))

In [ ]:
resume_id = 17108676

resume = annotation_df.loc[
    annotation_df["ID"] == resume_id,
    "Resume_str"
].iloc[0]

print(resume)

In [ ]:
import json

annotation = {
    "current_title": "Mechanical Engineering Intern",
    "previous_titles": [
        "Systems Engineering Intern",
        "Academic Employee (Grader)"
    ],
    "current_company": None,
    "previous_companies": [],
    "years_experience": None,
    "seniority": "junior",
    "primary_domain": "Mechanical Engineering",
    "industries": [
        "Engineering",
        "Biotechnology/Research"
    ],
    "core_skills": [
        "Mechanical Design",
        "CAD",
        "SolidWorks",
        "AutoCAD",
        "Robotics",
        "Energy Analysis",
        "Material Science",
        "Fluid Mechanics",
        "Heat Transfer",
        "Manufacturing",
        "Prototyping",
        "Project Management"
    ],
    "secondary_skills": [
        "Leadership",
        "Teamwork",
        "Communication",
        "Research",
        "Reporting",
        "Time Management",
        "Problem Solving",
        "Academic Mentoring"
    ],
    "tools": [
        "SolidWorks",
        "AutoCAD",
        "Microsoft Excel",
        "Microsoft Word",
        "Microsoft PowerPoint",
        "Gantt Chart",
        "Laser Cutter",
        "Automated Instrumentation"
    ],
    "leadership_experience": True,
    "key_achievements": [
        "Collaborated with a four-person team to build a mobile robot.",
        "Analyzed robot mobility, torque, friction, and energy transfer.",
        "Created CAD drawings and produced an acrylic clock using a laser cutter.",
        "Assisted with prototype assembly, manufacturing, and fabrication.",
        "Supervised approximately 20 kindergarten children as a teaching assistant."
    ],
    "location": None,
    "summary": (
        "Mechanical engineering student and Engineer-in-Training "
        "with experience in mechanical design, CAD, robotics, energy "
        "analysis, and prototyping. Worked on mobile robot development, "
        "CAD-based product design, manufacturing, and research operations. "
        "Demonstrates teamwork, project coordination, leadership, "
        "and communication skills."
    )
}

# Update only the selected resume
resume_id = 17108676

for field, value in annotation.items():
    annotation_df.loc[
        annotation_df["ID"] == resume_id, field
    ] = json.dumps(value) if isinstance(value, (list, dict)) else value

# Save updated annotations
OUTPUT_PATH = Path("../data/evaluation/resume_annotation_template.csv")

annotation_df.to_csv(OUTPUT_PATH, index=False)

print("First resume annotated and saved.")

In [ ]:
import json
from pathlib import Path

resume_id = 17108676

annotation = {
    "current_title": "Mechanical Engineering Intern",
    "previous_titles": [
        "Systems Engineering Intern",
        "Academic Employee (Grader)"
    ],
    "current_company": None,
    "previous_companies": [],
    "years_experience": None,
    "seniority": "junior",
    "primary_domain": "Mechanical Engineering",
    "industries": ["Engineering", "Biotechnology/Research"],
    "core_skills": [
        "Mechanical Design", "CAD", "Robotics",
        "Energy Analysis", "Material Science",
        "Fluid Mechanics", "Heat Transfer",
        "Manufacturing", "Prototyping", "Project Management"
    ],
    "secondary_skills": [
        "Leadership", "Teamwork", "Communication",
        "Research", "Reporting", "Time Management",
        "Problem Solving", "Academic Mentoring"
    ],
    "tools": [
        "SolidWorks", "AutoCAD", "Microsoft Excel",
        "Microsoft Word", "Microsoft PowerPoint",
        "Gantt Chart", "Laser Cutter",
        "Automated Instrumentation"
    ],
    "leadership_experience": True,
    "key_achievements": [
        "Collaborated with a four-person team to build a mobile robot.",
        "Analyzed robot mobility, torque, friction, and energy transfer.",
        "Created CAD drawings and produced an acrylic clock using a laser cutter.",
        "Assisted with prototype assembly, manufacturing, and fabrication.",
        "Supervised approximately 20 kindergarten children as a teaching assistant."
    ],
    "location": None,
    "summary": (
        "Mechanical engineering student and Engineer-in-Training "
        "with experience in mechanical design, CAD, robotics, energy "
        "analysis, and prototyping. Worked on mobile robot development, "
        "CAD-based product design, manufacturing, and research operations. "
        "Demonstrates teamwork, project coordination, leadership, "
        "and communication skills."
    )
}

# Update the selected record
mask = annotation_df["ID"] == resume_id

if mask.sum() != 1:
    raise ValueError("Expected exactly one matching resume.")

for field, value in annotation.items():
    annotation_df.loc[mask, field] = (
        json.dumps(value, ensure_ascii=False)
        if isinstance(value, list)
        else value
    )

# Save updated annotation file
output_path = Path("../data/evaluation/resume_annotation_template.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

annotation_df.to_csv(output_path, index=False)

print("Annotation saved.")
print("Annotated resumes:", annotation_df["current_title"].notna().sum())

In [ ]:
resume_id = annotation_df.loc[1, "ID"]

resume = annotation_df.loc[
    annotation_df["ID"] == resume_id,
    "Resume_str"
].iloc[0]

print("Resume ID:", resume_id)
print(resume)

In [ ]:
resume_id = 41950126

resume = annotation_df.loc[
    annotation_df["ID"] == resume_id,
    "Resume_str"
].iloc[0]

chunk_size = 2500

for start in range(0, len(resume), chunk_size):
    print(f"\n--- CHUNK {start // chunk_size + 1} ---\n")
    print(resume[start:start + chunk_size])

In [ ]:
import json
from pathlib import Path

resume_id = 41950126

annotation = {
    "current_title": "Loss Prevention / BSA Analyst",
    "previous_titles": [
        "Branch Representative",
        "Operations Supervisor",
        "Senior Operations Associate",
        "Teller Manager"
    ],
    "current_company": None,
    "previous_companies": [],
    "years_experience": None,
    "seniority": "senior",
    "primary_domain": "Banking Operations and Compliance",
    "industries": [
        "Banking",
        "Credit Union",
        "Financial Services"
    ],
    "core_skills": [
        "Banking Operations",
        "Anti-Money Laundering",
        "BSA Compliance",
        "Fraud Prevention",
        "Fraud Investigation",
        "Risk Management",
        "Regulatory Compliance",
        "Audit Procedures",
        "Operations Management",
        "Team Leadership",
        "Process Improvement",
        "Quality Control",
        "Customer Service",
        "Performance Management"
    ],
    "secondary_skills": [
        "Analytical Skills",
        "Communication",
        "Coaching",
        "Mentoring",
        "Decision Making",
        "Reporting",
        "Time Management",
        "Problem Solving"
    ],
    "tools": [
        "Epysis/Symitar",
        "Meridian Link",
        "STAR Network",
        "Patriot Officer",
        "VISA",
        "Microsoft Office",
        "Anti-Money Laundering Software"
    ],
    "leadership_experience": True,
    "key_achievements": [
        "Led and trained 12 junior team members while maintaining passing audits, 90% customer service, and SLAs for five consecutive years.",
        "Managed a team of 10 branch tellers.",
        "Mentored 18 junior team members.",
        "Balanced daily cash deposits and bank vault inventory with a zero error rate.",
        "Received a regional branch recognition award for outstanding customer service.",
        "Achieved an internal audit score of 90% for five consecutive years.",
        "Received multiple individual corporate recognitions."
    ],
    "location": None,
    "summary": (
        "Experienced banking professional with extensive experience in "
        "banking operations, BSA compliance, anti-money laundering, fraud "
        "prevention, and risk management. Has held roles in branch operations, "
        "financial investigations, and operational supervision. Demonstrates "
        "leadership through managing teams, mentoring employees, maintaining "
        "audit standards, and improving operational performance."
    )
}

mask = annotation_df["ID"] == resume_id

if mask.sum() != 1:
    raise ValueError("Expected exactly one matching resume.")

for field, value in annotation.items():
    annotation_df.loc[mask, field] = (
        json.dumps(value, ensure_ascii=False)
        if isinstance(value, list)
        else value
    )

output_path = Path("../data/evaluation/resume_annotation_template.csv")
annotation_df.to_csv(output_path, index=False)

print("Annotation saved successfully.")
print("Annotated records:", annotation_df["current_title"].notna().sum())

In [ ]:
from pathlib import Path

# Only select resumes that are not yet annotated
unannotated = annotation_df[
    annotation_df["current_title"].isna()
].copy()

BATCH_SIZE = 5

batch_dir = Path("../data/evaluation/annotation_batches")
batch_dir.mkdir(parents=True, exist_ok=True)

for batch_number, start in enumerate(
    range(0, len(unannotated), BATCH_SIZE), start=1
):
    batch = unannotated.iloc[start:start + BATCH_SIZE]

    output_file = batch_dir / f"batch_{batch_number:02d}.txt"

    with open(output_file, "w", encoding="utf-8") as file:
        for _, row in batch.iterrows():
            file.write(f"\n{'=' * 80}\n")
            file.write(f"RESUME ID: {row['ID']}\n")
            file.write(f"CATEGORY: {row['Category']}\n")
            file.write(f"{'=' * 80}\n\n")
            file.write(str(row["Resume_str"]))
            file.write("\n\n")

    print(f"Created {output_file.name}: {len(batch)} resumes")

print("\nTotal remaining:", len(unannotated))
print("Total batches:", (len(unannotated) + BATCH_SIZE - 1) // BATCH_SIZE)

In [ ]:
from pathlib import Path

batch_path = Path("../data/evaluation/annotation_batches/batch_01.txt")

content = batch_path.read_text(encoding="utf-8")

print(content)

In [ ]:
from pathlib import Path

batch_path = Path("../data/evaluation/annotation_batches/batch_01.txt")

content = batch_path.read_text(encoding="utf-8")

print(content)

In [ ]:
import json
from pathlib import Path

annotations = {
    34953092: {
        "current_title": "Corporate Banking Assistant Intern",
        "previous_titles": [
            "Office Assistant Intern",
            "Undergraduate Researcher",
            "Assistant Analyst Intern"
        ],
        "current_company": None,
        "previous_companies": [],
        "years_experience": None,
        "seniority": "junior",
        "primary_domain": "Financial Analysis and Banking",
        "industries": [
            "Banking",
            "Financial Services",
            "Research"
        ],
        "core_skills": [
            "Financial Analysis",
            "Corporate Banking",
            "Financial Forecasting",
            "Risk Analysis",
            "Cash Flow Analysis",
            "SQL",
            "Python",
            "Machine Learning",
            "Image Classification",
            "Database Development",
            "Statistical Modeling"
        ],
        "secondary_skills": [
            "Research",
            "Communication",
            "Recommendation Writing",
            "Data Analysis",
            "Problem Solving"
        ],
        "tools": [
            "C/C++", "Python", "MATLAB", "SQL", "R", "LUA", "VBA"
        ],
        "leadership_experience": True,
        "key_achievements": [
            "Developed a customer feedback system using VBA, improving team efficiency by 20%.",
            "Created image classification solutions that improved image processing speed by 30%.",
            "Developed an Updatable Documentation Notification System for commercial loan information management.",
            "Researched SMEs using SQL to support financial product recommendations."
        ],
        "location": None,
        "summary": (
            "Financial and computer engineering graduate with experience "
            "in corporate banking, financial analysis, database development, "
            "and machine learning research. Demonstrated skills in programming, "
            "financial risk analysis, image classification, and process improvement."
        )
    },

    16276121: {
        "current_title": "Digital Marketing Associate",
        "previous_titles": ["Account Manager"],
        "current_company": None,
        "previous_companies": [],
        "years_experience": None,
        "seniority": "mid",
        "primary_domain": "Digital Marketing",
        "industries": [
            "Healthcare",
            "Beauty and Cosmetics",
            "Automotive",
            "Education"
        ],
        "core_skills": [
            "Digital Marketing",
            "Social Media Marketing",
            "Account Management",
            "Brand Development",
            "Campaign Planning",
            "Campaign Execution",
            "Advertising",
            "Market Analysis",
            "Customer Acquisition",
            "Customer Relationship Management"
        ],
        "secondary_skills": [
            "Client Communication",
            "Strategic Planning",
            "Customer Service",
            "Teamwork",
            "Scheduling",
            "Relationship Management"
        ],
        "tools": [
            "Google Analytics",
            "Facebook",
            "Instagram",
            "Twitter",
            "Blog Platforms",
            "E-commerce Platforms",
            "Call Tracking Tools"
        ],
        "leadership_experience": None,
        "key_achievements": [
            "Managed online marketing accounts for multiple clients.",
            "Coordinated online campaign creation, approvals, and scheduling.",
            "Recognized by senior management for efficient client-agency relationship management."
        ],
        "location": None,
        "summary": (
            "Digital marketing professional with experience in campaign "
            "execution, account management, social media marketing, and "
            "client relationship management. Has worked with clients across "
            "healthcare, beauty, automotive, and education-related businesses."
        )
    }
}

for resume_id, annotation in annotations.items():
    mask = annotation_df["ID"] == resume_id

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one record for {resume_id}")

    for field, value in annotation.items():
        annotation_df.loc[mask, field] = (
            json.dumps(value, ensure_ascii=False)
            if isinstance(value, list)
            else value
        )

output_path = Path("../data/evaluation/resume_annotation_template.csv")
annotation_df.to_csv(output_path, index=False)

print("Annotations saved successfully.")
print(
    "Annotated resumes:",
    annotation_df["current_title"].notna().sum()
)
print("Remaining:", annotation_df["current_title"].isna().sum())

In [ ]:

import json
from pathlib import Path

annotations = {
    "85417107": {
        "current_title": "Hiring Manager / Recruiting Manager",
        "previous_titles": [
            "Recruiting Manager",
            "Senior Healthcare Recruiter",
            "Human Resource Specialist",
            "Human Resource Specialist (Temporary)",
            "Program Coordinator"
        ],
        "current_company": None,
        "previous_companies": [],
        "years_experience": "9+ years (self-reported)",
        "seniority": "senior",
        "primary_domain": "Healthcare Recruitment and Human Resources",
        "industries": [
            "Healthcare",
            "Social Services",
            "Staffing and Recruitment"
        ],
        "core_skills": [
            "Full-Cycle Recruitment",
            "Talent Acquisition",
            "Talent Management",
            "Healthcare Recruitment",
            "Recruitment Strategy",
            "Workforce Planning",
            "Candidate Sourcing",
            "Candidate Screening",
            "Interviewing",
            "Salary Negotiation",
            "Recruitment Team Management",
            "Business Development",
            "Account Management",
            "Training and Development",
            "Employee Relations",
            "Compliance"
        ],
        "secondary_skills": [
            "Coaching",
            "Communication",
            "Relationship Management",
            "Performance Evaluation",
            "Marketing",
            "Conflict Resolution",
            "Workforce Planning",
            "Client Management"
        ],
        "tools": [
            "Microsoft Office",
            "JobDiva ATS",
            "CareerBuilder",
            "LinkedIn Recruiter"
        ],
        "leadership_experience": True,
        "key_achievements": [
            "Managed and coached professional recruiting teams.",
            "Developed recruiting plans and sourcing strategies for clinical and non-clinical roles.",
            "Managed multiple healthcare recruitment accounts.",
            "Developed recruitment pipelines and training programs.",
            "Managed recruiting metrics and weekly requisition reporting."
        ],
        "location": "United States",
        "summary": (
            "Senior healthcare recruitment and HR professional with 9+ years "
            "of self-reported experience. Experienced in managing recruiting "
            "teams, full-cycle recruitment, healthcare staffing, workforce "
            "planning, business development, and talent management."
        )
    },

    "17570634": {
        "current_title": "Owner / IT Support Business Manager",
        "previous_titles": [
            "Commercial Project Manager and Construction Manager",
            "Construction Manager",
            "Project Manager and Network Consultant"
        ],
        "current_company": None,
        "previous_companies": [],
        "years_experience": None,
        "seniority": "senior",
        "primary_domain": "Construction and Project Management",
        "industries": [
            "Construction",
            "Information Technology",
            "Telecommunications",
            "Healthcare IT"
        ],
        "core_skills": [
            "Project Management",
            "Construction Management",
            "Commercial Construction",
            "Budget Management",
            "Scheduling",
            "Resource Planning",
            "Team Management",
            "Contractor Coordination",
            "Network Design",
            "Network Administration",
            "Technical Support",
            "Network Security",
            "Electronic Medical Records",
            "Process Improvement",
            "Quality Control",
            "Vendor Management"
        ],
        "secondary_skills": [
            "Leadership",
            "Communication",
            "Problem Solving",
            "Customer Relationship Management",
            "Marketing",
            "Training",
            "Human Resources",
            "Procurement"
        ],
        "tools": [
            "Check Point Security Software",
            "Electronic Medical Records Systems",
            "Scheduling Software",
            "Wireless Networking",
            "Computer Hardware",
            "Computer Networks"
        ],
        "leadership_experience": True,
        "key_achievements": [
            "Contributed to completion of 21 commercial projects valued at approximately $50 million.",
            "Managed multiple smaller projects with a combined value of approximately $10 million.",
            "Coordinated approximately 100 subcontractors and skilled trade workers.",
            "Led project crews of up to 20 individuals.",
            "Contributed to annual new-home sales exceeding $25 million.",
            "Increased sales by over 20% through targeted software marketing.",
            "Saved over $10,000 annually through telecommuting procedures."
        ],
        "location": "Chicago area, Illinois, USA",
        "summary": (
            "Experienced project and construction manager with additional "
            "experience owning and operating an IT support business. Skilled "
            "in commercial construction projects, team leadership, budgeting, "
            "scheduling, network administration, technical support, and "
            "process improvement."
        )
    },

    "33685988": {
        "current_title": "Finance Officer",
        "previous_titles": [
            "Payroll Assistant",
            "Accounts Officer",
            "Supervisor and Senior Accounts Clerk"
        ],
        "current_company": None,
        "previous_companies": [],
        "years_experience": None,
        "seniority": "senior",
        "primary_domain": "Finance and Accounting",
        "industries": [
            "Finance",
            "Accounting",
            "Banking"
        ],
        "core_skills": [
            "Financial Reporting",
            "General Ledger",
            "Bank Reconciliation",
            "Account Reconciliation",
            "Investment Accounting",
            "Budget Analysis",
            "Cash Flow Analysis",
            "Payroll Processing",
            "Audit Preparation",
            "Financial Analysis",
            "Inventory Reconciliation",
            "Accounts Payable",
            "Staff Supervision",
            "Accounting Journals"
        ],
        "secondary_skills": [
            "Analytical Reasoning",
            "Leadership",
            "Team Building",
            "Training",
            "Communication",
            "Customer Service",
            "Conflict Resolution",
            "Process Improvement",
            "Problem Solving"
        ],
        "tools": [
            "ACCPAC",
            "Turbo Pay"
        ],
        "leadership_experience": True,
        "key_achievements": [
            "Investigated and resolved errors in Savings and Interest Payable schedules.",
            "Streamlined daily reconciliation procedures.",
            "Supervised a team of eleven employees.",
            "Trained staff on standard operating procedures.",
            "Improved inventory and reconciliation procedures.",
            "Prepared cash flow statements and scheduled financial reports.",
            "Maintained timely preparation of monthly and annual returns."
        ],
        "location": "Jamaica",
        "summary": (
            "Finance and accounting professional experienced in financial "
            "reporting, general ledger operations, reconciliations, investment "
            "accounting, budget analysis, payroll, audit preparation, and "
            "supervising accounting staff."
        )
    },

    "31677347": {
        "current_title": "Software Engineering Manager",
        "previous_titles": [
            "Principal Software Engineer",
            "Software Engineer"
        ],
        "current_company": None,
        "previous_companies": [],
        "years_experience": None,
        "seniority": "senior",
        "primary_domain": "Software Engineering and Media Systems",
        "industries": [
            "Software Development",
            "Broadcast Media",
            "Video Technology",
            "Media Production"
        ],
        "core_skills": [
            "Software Architecture",
            "C++ Development",
            "Object-Oriented Programming",
            "API Design",
            "REST API Development",
            "Microservices and SOA",
            "Multithreading",
            "Network Programming",
            "Video Processing",
            "Audio and Video Rendering",
            "Media Server Development",
            "Team Management",
            "Technical Leadership",
            "System Design",
            "Performance Optimization",
            "Agile Development",
            "Scrum"
        ],
        "secondary_skills": [
            "Technical Documentation",
            "Code Review",
            "Mentoring",
            "Problem Solving",
            "Troubleshooting",
            "Project Planning",
            "Cross-Team Collaboration",
            "Process Improvement"
        ],
        "tools": [
            "Visual C++",
            "C++",
            "Win32",
            "MFC",
            "ATL",
            "Mercurial",
            "Jenkins",
            "RapidJSON",
            "JavaScript",
            "C#",
            "HTML5",
            "MPEG-DASH",
            "TCP",
            "FTP",
            "XML",
            "JSON"
        ],
        "leadership_experience": True,
        "key_achievements": [
            "Led engineering teams across the United States and Beijing, including direct and indirect reports.",
            "Managed development of more than 30 video server products and applications.",
            "Architected and implemented a service-oriented architecture with web services and additional services.",
            "Developed a C++ engine for segmented MP4 delivery supporting H.264 and AAC.",
            "Published REST API and JSON protocol specifications.",
            "Optimized asset management to maintain UI performance under a few milliseconds.",
            "Received a Reuse Innovation Award for intellectual property reuse."
        ],
        "location": None,
        "summary": (
            "Senior software engineering leader specializing in C++, software "
            "architecture, video and media systems, APIs, distributed services, "
            "network programming, and engineering team management. Experienced "
            "in developing and scaling professional broadcast video products."
        )
    },

    "59696315": {
        "current_title": "Business Development Director",
        "previous_titles": [
            "Director of Business Development",
            "Business Development Executive",
            "Director of Client Development",
            "Director of Sales and Marketing",
            "Account Executive",
            "Territory Manager"
        ],
        "current_company": None,
        "previous_companies": [],
        "years_experience": "20+ years (self-reported)",
        "seniority": "executive",
        "primary_domain": "Business Development and Sales",
        "industries": [
            "Digital Marketing",
            "Advertising",
            "Media",
            "Marketing Services",
            "Transportation and Logistics"
        ],
        "core_skills": [
            "Business Development",
            "Enterprise Sales",
            "Strategic Sales Planning",
            "Lead Generation",
            "Client Acquisition",
            "Account Management",
            "Relationship Management",
            "Sales Strategy",
            "Negotiation",
            "Proposal Development",
            "Revenue Growth",
            "Market Research",
            "Digital Marketing",
            "Brand Strategy",
            "Client Retention",
            "Business Strategy"
        ],
        "secondary_skills": [
            "Leadership",
            "Communication",
            "Presentation Skills",
            "Coaching",
            "Project Management",
            "Research",
            "Report Writing",
            "Problem Solving",
            "Networking",
            "Team Collaboration"
        ],
        "tools": [
            "LinkedIn Recruiter",
            "Web Analytics",
            "SEO",
            "SEM",
            "Social Media Platforms",
            "Digital Marketing Platforms"
        ],
        "leadership_experience": True,
        "key_achievements": [
            "Closed $2.5 million in sales within the first calendar year at Armchair Media.",
            "Closed more than $2 million in sales within the first calendar year at LBi Atlanta.",
            "Achieved a peak annual sales figure of $3.5 million.",
            "Secured clients including CNN, Turner, Coca-Cola, Disney, and The College Board.",
            "Acquired clients including UPS, Home Depot, ING Financial Services, and Cox Communications.",
            "Developed business growth strategies and trained consultants in account development."
        ],
        "location": "Atlanta, Georgia, USA (based on employment locations)",
        "summary": (
            "Senior business development and sales professional with 20+ years "
            "of self-reported relationship-building experience. Experienced "
            "in enterprise client acquisition, digital marketing services, "
            "strategic sales, account management, negotiation, and revenue growth."
        )
    }
}

# Verify that all five resume IDs exist.
id_column = "ID"
existing_ids = set(annotation_df[id_column].astype(str))
missing_ids = set(annotations) - existing_ids

if missing_ids:
    raise ValueError(f"These resume IDs were not found: {missing_ids}")

# Verify that all annotation fields exist.
annotation_fields = list(next(iter(annotations.values())).keys())
missing_columns = set(annotation_fields) - set(annotation_df.columns)

if missing_columns:
    raise ValueError(f"Missing annotation columns: {missing_columns}")

# Update the existing DataFrame.
for resume_id, values in annotations.items():
    row_index = annotation_df.index[
        annotation_df[id_column].astype(str) == resume_id
    ][0]

    for field, value in values.items():
        if isinstance(value, list):
            value = json.dumps(value, ensure_ascii=False)

        annotation_df.at[row_index, field] = value

# Save the updated annotation file.
output_path = Path("../data/evaluation/resume_annotation_template.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

annotation_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Batch 02 annotations saved successfully.")
print("Newly annotated resumes:", len(annotations))
print("Output:", output_path.resolve())


In [ ]:

import json
from pathlib import Path

resume_id = "24655918"

annotation = {
    "current_title": "Senior Graphic Designer",
    "previous_titles": [
        "Senior Graphic Designer / Production Manager",
        "Graphic Designer"
    ],
    "current_company": None,
    "previous_companies": [],
    "years_experience": "30+ years (self-reported)",
    "seniority": "senior",
    "primary_domain": "Graphic Design and Visual Communications",
    "industries": [
        "Graphic Design",
        "Print and Signage",
        "Advertising and Marketing"
    ],
    "core_skills": [
        "Graphic Design",
        "Logo Design",
        "Brand Design",
        "Corporate Design",
        "Print Design",
        "Signage Design",
        "Conceptual Design",
        "Production Management",
        "Creative Project Management",
        "Quality Assurance",
        "Visual Communication"
    ],
    "secondary_skills": [
        "Team Leadership",
        "Training and Development",
        "Strategic Planning",
        "Client Relationship Management",
        "Vendor Coordination",
        "Customer Service",
        "Creative Problem Solving",
        "Project Coordination"
    ],
    "tools": [
        "Adobe Illustrator",
        "Adobe Photoshop",
        "CorelDRAW",
        "Omega Composer",
        "QuickBooks",
        "Microsoft Office",
        "Keyed IN",
        "Pace Software",
        "HP 5500 Printer",
        "HP DesignJet L25500",
        "Summa D120R",
        "Graphtec",
        "Ioline Graphics Plotter",
        "FC7500 Plotter"
    ],
    "leadership_experience": True,
    "key_achievements": [
        "Managed graphic design and production projects from concept through final delivery.",
        "Built client relationships and grew the client base through strategic relationship building.",
        "Mentored staff on customer service initiatives.",
        "Coordinated design projects and worked with vendors and outside installers.",
        "Created logos, dimensional signs, banners, trade show graphics, and other branded materials."
    ],
    "location": None,
    "summary": (
        "Senior graphic designer with over 30 years of self-reported experience "
        "in print, logo design, dimensional signage, and brand-building design. "
        "Experienced in production management, Adobe design software, client "
        "coordination, and creative project delivery."
    )
}

# Confirm the resume exists.
matches = annotation_df.index[
    annotation_df["ID"].astype(str) == resume_id
]

if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one row for {resume_id}, found {len(matches)}"
    )

# Confirm the schema matches.
missing_columns = set(annotation) - set(annotation_df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Update only this resume.
row_index = matches[0]

for field, value in annotation.items():
    if isinstance(value, list):
        value = json.dumps(value, ensure_ascii=False)

    annotation_df.at[row_index, field] = value

# Save the completed annotation file.
output_path = Path("../data/evaluation/resume_annotation_template.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

annotation_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final annotation saved successfully.")
print("Resume ID:", resume_id)
print("Total annotation target: 10")
print("Output:", output_path.resolve())
